In [ ]:
import re

import pandas as pd
import requests
from tqdm import tqdm

def project_exists_in_gdc(project_id: str) -> bool:
    response = requests.get(f"https://api.gdc.cancer.gov/projects/{project_id}", params={"fields": "project_id"}, timeout=10)
    if response.status_code == 200:
        data = response.json()
        return "data" in data and "project_id" in data["data"]
    elif response.status_code == 404:
        return False
    else:
        response.raise_for_status()
        return False

In [ ]:
df = pd.read_csv("questions-v1.csv")

In [ ]:
ssm_cnv_templates = [
    r"What percentage of cancers have simple somatic mutations and copy number variants in (?P<gene>.+?) in the genomic data commons for (?P<project>.+?) project\?",
    r"What is the incidence of simple somatic mutations and copy number variants in (?P<gene>.+?) in the genomic data commons for (?P<project>.+?) project \?",
    r"How common are simple somatic mutations and copy number variants in (?P<gene>.+?) in (?P<project>.+?) project in the genomic data commons\?",
    r"What fraction of cases have simple somatic mutations and copy number variants in (?P<gene>.+?) in (?P<project>.+?) project in the genomic data commons\?",
    r"What proportion of cancer patients exhibit simple somatic mutations and copy number variants in (?P<gene>.+?) in (?P<project>.+?) project in the genomic data commons\?",
    r"What is the frequency of simple somatic mutations and copy number variants in (?P<gene>.+?) in (?P<project>.+?) project in the genomic data commons\?",
]
HGVS = r"[A-Z]\d+(?:[A-Z](?:fs\*\d+)?|\*)" # missense, nonsense, frameshift
ssm_change_templates = [
    rf"How often is the (?P<gene>.+?) (?P<hgvs>{HGVS}?) found in (?P<project>.+?) project in the genomic data commons\?",
    rf"What is the occurrence rate of (?P<gene>.+?) (?P<hgvs>{HGVS}?) in (?P<project>.+?) project in the genomic data commons \?",
    rf"What is the rate of occurrence of (?P<gene>.+?) (?P<hgvs>{HGVS}?) mutation in (?P<project>.+?) project in the genomic data commons\?",
    rf"How frequently are (?P<gene>.+?) (?P<hgvs>{HGVS}?) mutations detected in (?P<project>.+?) project in the genomic data commons\?",
    rf"What is the frequency of (?P<gene>.+?) (?P<hgvs>{HGVS}?) in (?P<project>.+?) project in the genomic data commons\?",
]
CNV_CHANGE = r"heterozygous deletion|homozygous deletion|gain|amplification"
cnv_change_templates = [
    rf"What is the frequency of somatic (?P<gene>.+?) (?P<cnv>{CNV_CHANGE}) in (?P<project>.+?) project in the genomic data commons\?",
    rf"What is the incidence of somatic (?P<gene>.+?) (?P<cnv>{CNV_CHANGE}) in (?P<project>.+?) project in the genomic data commons\?",
    rf"Can you provide the frequency of (?P<gene>.+?) (?P<cnv>{CNV_CHANGE}) in (?P<project>.+?) project in the genomic data commons\?",
    rf"In (?P<project>.+?) project data from the genomic data commons, what is the frequency of (?P<gene>.+?) (?P<cnv>{CNV_CHANGE})\?",
]
msi_templates = [
    r"How common is microsatellite instability in (?P<project>.+?) project cases in the genomic data commons\?",
    r"Can you provide the prevalence of microsatellite instability in (?P<project>.+?) project in the genomic data commons\?",
    r"What percentage of (?P<project>.+?) project patients have microsatellite instability in the genomic data commons\?",
    r"How often is microsatellite instability observed in (?P<project>.+?) project in the genomic data commons\?",
    r"In (?P<project>.+?) project, what is the occurrence rate of microsatellite instability in the genomic data commons\?",
    r"What is the incidence of microsatellite instability in (?P<project>.+?) project in the genomic data commons\?",
    r"What is the frequency of microsatellite instability in (?P<project>.+?) project in the genomic data commons\?",
]
cooc_templates = [
    rf"What is the co-occurence frequency of somatic (?P<cnv>{CNV_CHANGE})s in (?P<gene_1>.+?) and (?P<gene_2>.+?) in the (?P<project>.+?)(?: project)? in the genomic data commons\?",
]

redo_templates = [
    lambda vals: f"What percentage of cancers have simple somatic mutations and copy number variants in {vals['gene']} in the genomic data commons for the {vals['project']} project?",
    lambda vals: f"What is the incidence of simple somatic mutations and copy number variants in {vals['gene']} in the genomic data commons for the {vals['project']} project?",
    lambda vals: f"How common are simple somatic mutations and copy number variants in {vals['gene']} in the {vals['project']} project in the genomic data commons?",
    lambda vals: f"What fraction of cases have simple somatic mutations and copy number variants in {vals['gene']} in the {vals['project']} project in the genomic data commons?",
    lambda vals: f"What proportion of cancer patients exhibit simple somatic mutations and copy number variants in {vals['gene']} in the {vals['project']} project in the genomic data commons?",
    lambda vals: f"What is the frequency of simple somatic mutations and copy number variants in {vals['gene']} in the {vals['project']} project in the genomic data commons?",
    lambda vals: f"How often is the {vals['gene']} {vals['hgvs']} found in the {vals['project']} project in the genomic data commons?",
    lambda vals: f"What is the occurrence rate of {vals['gene']} {vals['hgvs']} in the {vals['project']} project in the genomic data commons?",
    lambda vals: f"What is the rate of occurrence of {vals['gene']} {vals['hgvs']} mutation in the {vals['project']} project in the genomic data commons?",
    lambda vals: f"How frequently are {vals['gene']} {vals['hgvs']} mutations detected in the {vals['project']} project in the genomic data commons?",
    lambda vals: f"What is the frequency of {vals['gene']} {vals['hgvs']} in the {vals['project']} project in the genomic data commons?",
    lambda vals: f"What is the frequency of somatic {vals['gene']} {vals['cnv']} in the {vals['project']} project in the genomic data commons?",
    lambda vals: f"What is the incidence of somatic {vals['gene']} {vals['cnv']} in the {vals['project']} project in the genomic data commons?",
    lambda vals: f"Can you provide the frequency of {vals['gene']} {vals['cnv']} in the {vals['project']} project in the genomic data commons?",
    lambda vals: f"In the {vals['project']} project data from the genomic data commons, what is the frequency of {vals['gene']} {vals['cnv']}?",
    lambda vals: f"How common is microsatellite instability in the {vals['project']} project cases in the genomic data commons?",
    lambda vals: f"Can you provide the prevalence of microsatellite instability in the {vals['project']} project in the genomic data commons?",
    lambda vals: f"What percentage of the {vals['project']} project patients have microsatellite instability in the genomic data commons?",
    lambda vals: f"How often is microsatellite instability observed in the {vals['project']} project in the genomic data commons?",
    lambda vals: f"In the {vals['project']} project, what is the occurrence rate of microsatellite instability in the genomic data commons?",
    lambda vals: f"What is the incidence of microsatellite instability in the {vals['project']} project in the genomic data commons?",
    lambda vals: f"What is the frequency of microsatellite instability in the {vals['project']} project in the genomic data commons?",
    lambda vals: f"What is the co-occurrence frequency of somatic {vals['cnv']}s in {vals['gene_1']} and {vals['gene_2']} in the {vals['project']}(?: project)? in the genomic data commons?",
]

In [ ]:
patterns = [
    (cat, re.compile(template))
    for cat, templates in enumerate([
        ssm_cnv_templates,
        ssm_change_templates,
        cnv_change_templates,
        msi_templates,
        cooc_templates,
    ]) for template in templates
]

In [ ]:
def match_pattern(x):
    ret = {"cat": -1}
    for global_pat_num, (cat, pat) in enumerate(patterns):
        if (matched := pat.match(x)) is not None:
            ret["cat"] = cat
            ret["pat_num"] = global_pat_num
            for k, v in matched.groupdict().items():
                ret[k] = v
    return ret

In [ ]:
extracted = pd.DataFrame([match_pattern(x) for x in df["questions"]])

In [ ]:
extracted["cat"].value_counts()

In [ ]:
def clean_project(xs: pd.Series) -> pd.Series:
    xs = xs.str.split().str[-1]
    for project in tqdm(set(xs)):
        assert project_exists_in_gdc(project), f"Project {project} Not Found!"
    return xs

In [ ]:
extracted["project"] = clean_project(extracted["project"])

In [ ]:
cat_dfs = {cat: cat_df for cat, cat_df in extracted.groupby("cat")}

In [ ]:
all_cols = ['gene', 'cnv', 'project', 'hgvs', 'gene_1', 'gene_2']
def require_all(df, _cols):
    not_cols = set(all_cols) - set(_cols)
    cols = set(all_cols) & set(_cols)
    assert cols == set(_cols), f"Unexpected columns: {set(_cols) - set(all_cols)}"
    for col in cols:
        temp = df[df[col].isna()]
        if len(temp) != 0:
            print(f"Found NaN values in {col}")
            return temp
    for col in not_cols:
        temp = df[~df[col].isna()]
        if len(temp) != 0:
            print(f"Found unexpected values in {col}")
            return temp
    return None

In [ ]:
for cat, req_cols in enumerate([
    ["gene", "project"], # ssm and cnv
    ["gene", "hgvs", "project"], # ssm
    ["gene", "cnv", "project"], # cnv
    ["project"], # msi
    ["gene_1", "gene_2", "cnv", "project"], # cooc
]):
    print(f"checking that cat {cat} has stricly all values for {req_cols}")
    temp = require_all(cat_dfs[cat], req_cols)
    if temp is not None:
        break

In [ ]:
redos = []
for i, row in extracted.iterrows():
    redos.append(redo_templates[row["pat_num"]](row))

In [ ]:
extracted["questions"] = redos

In [ ]:
extracted.to_csv("questions.csv", index=False)